# Module 04: Evolutionary Computation, Genetic Algorithms & Edge of Chaos

**Core Mathematical & Algorithmic Framework:**
- **Density Classification Task (DCT):** $N = 149$ periodic 1D CA with radius $r = 3$ encoded as a $2^7 = 128$-bit GA chromosome.
- **GKL Particle Rule ($r=3$):** Propagates checkerboard signals ($v = +1/3$) and $0$-domain walls ($v = -1$) to resolve global majority.
- **Holland's Schema Theorem:** $\mathbb{E}[m(H, t+1)] \ge m(H, t)\frac{f(H)}{\bar{f}}\left[1 - p_c\frac{\delta(H)}{L-1}\right](1-p_m)^{o(H)}$.

In [ ]:
import subprocess
import sys
from pathlib import Path

def _ensure_toolkit_on_path():
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd.parent, cwd.parent.parent):
        if (cand / "core").is_dir():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            return cand
    clone_dir = cwd / "The-Collective-Intelligence-Toolkit"
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/poria-azadi96/The-Collective-Intelligence-Toolkit.git", str(clone_dir)],
            check=True,
        )
    if str(clone_dir) not in sys.path:
        sys.path.insert(0, str(clone_dir))
    return clone_dir

REPO_ROOT = _ensure_toolkit_on_path()
import matplotlib.pyplot as plt
import ipywidgets as widgets
from core.evolutionary_ga import (
    create_evolutionary_dashboard_figure,
    holland_schema_lower_bound,
    run_module04_tier1,
    run_module04_tier2,
    run_module04_tier3,
    schema_properties,
)
from core.real_world_demos import (
    demo_m04_automl_nas,
    demo_m04_protein_directed_evolution,
    demo_m04_swarm_robotics_foraging,
)

def run_evolutionary_lab(
    checkpoint='Epoch 3/4 (GKL)',
    init_density=0.54,
    run_short_ga=False,
):
    fig, stats = create_evolutionary_dashboard_figure(
        checkpoint=checkpoint,
        init_density=init_density,
        run_short_ga=run_short_ga,
        seed=7,
    )
    plt.show()
    o_h, d_h = schema_properties('1**0*1**')
    schema_next = holland_schema_lower_bound(m_t=4, f_schema=0.78, f_avg=0.52, defining_length=d_h, order_k=o_h)
    print(
        f"Selected Rule: {stats['checkpoint']} | Uniform Acc: {stats['fit_uniform']:.2%} | "
        f"Unbiased Acc: {stats['fit_unbiased']:.2%} | Final Density: {stats['final_density']:.2f}"
    )
    print(
        f"Holland Schema Theorem H=1**0*1** (o(H)={o_h}, delta={d_h}, m_t=4, f(H)/f_avg=1.50) "
        f"=> E[m(H, t+1)] >= {schema_next:.2f}"
    )

widgets.interact(
    run_evolutionary_lab,
    checkpoint=widgets.Dropdown(
        options=['Epoch 1', 'Epoch 2 (Asymmetric Block Expander)', 'Epoch 3/4 (GKL)'],
        value='Epoch 3/4 (GKL)',
        description='Checkpoint:',
    ),
    init_density=widgets.FloatSlider(min=0.20, max=0.80, step=0.02, value=0.54, description='Init Density:'),
    run_short_ga=widgets.Checkbox(value=False, description='Run Live 12-Gen GA'),
);

## 3-Tier Experimental Protocol
- **Tier 1 (Observation):** Switch `Epoch 1 -> Epoch 2 -> Epoch 3/4 (GKL)` at $\rho_0 = 0.54$ to compare frozen blocks vs. particle signals.
- **Tier 2 (Symmetry & Bias Analysis):** Test `Epoch 2` at $\rho_0 = 0.44$ vs. $0.56$ and plot accuracy across densities (Land-Belew boundary).
- **Tier 3 (Live Evolution & Schema Check):** Run `Live 12-Gen GA (Seed=7)` and verify Holland's Schema lower bound vs. observed counts.

In [ ]:
def run_m04_tier_selector(tier='Tier 1: Innovation Epochs on rho=0.54'):
    if tier.startswith('Tier 1'):
        fig, res = run_module04_tier1(init_density=0.54)
        print(res)
    elif tier.startswith('Tier 2'):
        fig, res = run_module04_tier2()
        print('Epoch 2 rho=0.44 final:', res['epoch2_rho044_final'], '| rho=0.56 final:', res['epoch2_rho056_final'])
    else:
        fig, res = run_module04_tier3(seed=7)
        print('Canonical Schema H=1**0*1**:', res['canonical_schema_example'])
        for rec in res['schema_records'][:5]:
            print(rec)
    plt.show()

widgets.interact(
    run_m04_tier_selector,
    tier=widgets.Dropdown(
        options=[
            'Tier 1: Innovation Epochs on rho=0.54',
            'Tier 2: Epoch 2 Symmetry Bias (0.44 vs 0.56)',
            'Tier 3: Live 12-Gen GA & Schema Check (Seed=7)',
        ],
        value='Tier 1: Innovation Epochs on rho=0.54',
        description='Protocol:',
    ),
);

## Module 04 Real-World Applications
1. **Computational Biology & Protein Design:** Directed evolution on Kauffman $NK$ rugged landscapes.
2. **Artificial Intelligence & AutoML:** Discrete Neural Architecture Search (Accuracy vs. FLOPs Pareto front).
3. **Swarm & Evolutionary Robotics:** Decentralized 2D foraging robots without a central controller.

In [ ]:
def run_m04_real_world_demo(application='1. Directed Protein Evolution (NK)'):
    if application.startswith('1.'):
        fig, res = demo_m04_protein_directed_evolution()
    elif application.startswith('2.'):
        fig, res = demo_m04_automl_nas()
    else:
        fig, res = demo_m04_swarm_robotics_foraging()
    plt.show()
    print({k: v for k, v in res.items() if not k.startswith('snaps')})

widgets.interact(
    run_m04_real_world_demo,
    application=widgets.Dropdown(
        options=[
            '1. Directed Protein Evolution (NK)',
            '2. AutoML Neural Architecture Search',
            '3. Swarm Robotics Foraging Evolution',
        ],
        value='1. Directed Protein Evolution (NK)',
        description='Case Study:',
    ),
);